In [1]:
!pip install mxnet d2l -q

In [55]:
!wget https://raw.githubusercontent.com/yunuscvlk/netflix-prize-data-analysis/refs/heads/main/testdata/random10ksample.csv
!wget https://raw.githubusercontent.com/yunuscvlk/netflix-prize-data-analysis/refs/heads/main/testdata/random50ksample.csv
!wget https://raw.githubusercontent.com/yunuscvlk/netflix-prize-data-analysis/refs/heads/main/testdata/random100ksample.csv
!wget https://raw.githubusercontent.com/yunuscvlk/netflix-prize-data-analysis/refs/heads/main/testdata/random1msample.csv

--2024-12-17 17:28:26--  https://raw.githubusercontent.com/yunuscvlk/netflix-prize-data-analysis/refs/heads/main/testdata/random10ksample.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16037 (16K) [text/plain]
Saving to: ‘random10ksample.csv’

random10ksample.csv 100%[===================>]  15.66K  --.-KB/s    in 0s      

2024-12-17 17:28:27 (40.9 MB/s) - ‘random10ksample.csv’ saved [16037/16037]

--2024-12-17 17:28:27--  https://raw.githubusercontent.com/yunuscvlk/netflix-prize-data-analysis/refs/heads/main/testdata/random50ksample.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... conne

In [76]:
import os
import pandas as pd

from d2l import mxnet as d2l
from mxnet import gluon, np, npx

npx.set_np()

def read_data(filename):
    data = pd.read_csv(filename, sep=';')

    num_customers = data["CustomerID"].unique().shape[0]
    num_movies = data["MovieID"].unique().shape[0]

    return data, num_customers, num_movies

def split_data(data, num_customers, num_movies, test_ratio=0.2):
    # Rastgele bir maske oluşturulur. %80'ı eğitim, %20'u test için ayrılır.
    mask = [True if x == 1 else False for x in np.random.uniform(0, 1, (len(data))) < 1 - test_ratio]

    # Eğitim verisinin maskesi ters çevrilerek test verisi maskesi oluşturulur.
    neg_mask = [not x for x in mask]

    # Maskelere göre eğitim ve test veri kümeleri ayrılır.
    train_data, test_data = data[mask], data[neg_mask]

    return train_data, test_data

def load_data(data, num_customers, num_movies):
    customers, movies, rates = [], [], []

    # Her satır için kullanıcı, öğe ve puan bilgileri çıkarılır.
    for line in data.itertuples():
        # Customer ve movie için dizini 0 tabanlı yapar, rate bilgisi alınır.
        customer_index, movie_index, rate = int(line[1] - 1), int(line[2] - 1), int(line[3])

        customers.append(customer_index)
        movies.append(movie_index)
        rates.append(rate)

    return np.array(customers), np.array(movies), np.array(rates)

# Veri kümesini eğitim ve test olarak bölüp, mini topluluklar (batch) halinde yükler.
def split_and_load(test_ratio=0.2, batch_size=256):
    data, num_customers, num_movies = read_data('random100ksample.csv')

    train_data, test_data = split_data(data, num_customers, num_movies, test_ratio)

    train_customers, train_movies, train_rates = load_data(train_data, num_customers, num_movies)
    test_customers, test_movies, test_rates = load_data(test_data, num_customers, num_movies)

    train_set = gluon.data.ArrayDataset(train_customers, train_movies, train_rates)
    test_set = gluon.data.ArrayDataset(test_customers, test_movies, test_rates)

    train_iter = gluon.data.DataLoader(train_set, shuffle=True, batch_size=batch_size)
    test_iter = gluon.data.DataLoader(test_set, batch_size=batch_size)

    return num_customers, num_movies, train_iter, test_iter


test_ratio, batch_size = 0.2, 256
num_customers, num_movies, train_iter, test_iter = split_and_load(test_ratio, batch_size)

In [77]:
from mxnet.gluon import nn

# Matris Faktörizasyon modelini tanımlayan bir sınıf
class MatrixFactorization(nn.Block):
    def __init__(self, num_customers, num_movies, num_factors, **kwargs):
        super(MatrixFactorization, self).__init__(**kwargs)

        # Customer'ların faktör temsillerini (embedding) tutar.
        self.customer_embedding = nn.Embedding(num_customers, num_factors)

        # Movie'lerin faktör temsillerini (embedding) tutar.
        self.movie_embedding = nn.Embedding(num_movies, num_factors)

        # Customer bazlı sapma (bias) değerlerini tutar.
        self.customer_bias = nn.Embedding(num_customers, 1)

        # Movie bazlı sapma (bias) değerlerini tutar.
        self.movie_bias = nn.Embedding(num_movies, 1)

    # İleri yönlü hesaplama (forward pass)
    def forward(self, customers, movies):
        # Customer'ların faktör temsillerini alır.
        customer_vecs = self.customer_embedding(customers)

        # Movie'lerin faktör temsillerini alır.
        movie_vecs = self.movie_embedding(movies)

        # Customer ve movie faktörleri arasında iç çarpımı hesaplar.
        preds = (
              (customer_vecs * movie_vecs).sum(axis=1) +
              self.customer_bias(customers).squeeze() + # Customer sapmalarını ekler.
              self.movie_bias(movies).squeeze() # Movie sapmalarını ekler.
            )

        return preds

In [78]:
from mxnet import autograd, nd, init
from mxnet.gluon import Trainer, loss as gloss

def train_recommender(net, train_iter, test_iter, num_epochs, lr, wd, ctx):
    # Modeli normal bir dağılımla başlat (standart sapma = 0.01)
    net.initialize(ctx=ctx, force_reinit=True, init=init.Normal(0.01))

    # Adam optimizasyon algoritması kullanılarak modelin parametrelerini eğitmek için bir trainer oluştur.
    trainer = Trainer(net.collect_params(), 'adam', {'learning_rate': lr, 'wd': wd})

    # L2Loss, tahmin ve gerçek değerler arasındaki kareler farkının ortalamasını alır.
    loss = gloss.L2Loss()

    for epoch in range(num_epochs):
        # Birikimli istatistikler için bir sayaç (toplam kayıp, örnek sayısı, batch sayısı)
        metric = d2l.Accumulator(3)

        for i, (customers, movies, rates) in enumerate(train_iter):
            # Verileri eğitim için belirtilen cihaza (CPU/GPU) taşır.
            customers, movies, rates = customers.as_in_ctx(ctx), movies.as_in_ctx(ctx), rates.as_in_ctx(ctx)
            with autograd.record():
                # Modelin tahminlerini hesapla
                preds = net(customers, movies)

                # Tahminler ile gerçek değerler arasındaki kaybı hesapla
                l = loss(preds, rates)

            # Kaybın türevlerini hesapla (geri yayılım)
            l.backward()

            # Optimizer ile modelin ağırlıklarını güncelle
            trainer.step(batch_size=rates.shape[0])

            # Toplam kayıp, veri örneklerinin sayısı ve batch sayısını güncelle
            metric.add(l.sum().item(), rates.size, 1)

        # RMSE'yi (Kök Ortalama Kare Hata) hesapla
        train_rmse = nd.sqrt(nd.array([metric[0]]) / nd.array([metric[1]]))
        print(f'epoch {epoch + 1}, train RMSE: {train_rmse.asscalar():.4f}')

In [79]:
ctx = d2l.try_gpu()
num_factors = 20  # Embedding boyutu
net = MatrixFactorization(num_customers, num_movies, num_factors)
num_epochs, lr, wd, batch_size = 20, 0.005, 0, 256
train_recommender(net, train_iter, test_iter, num_epochs, lr, wd, ctx)

epoch 1, train RMSE: 1.4912
epoch 2, train RMSE: 0.8146
epoch 3, train RMSE: 0.7565
epoch 4, train RMSE: 0.7390
epoch 5, train RMSE: 0.7287
epoch 6, train RMSE: 0.7210
epoch 7, train RMSE: 0.7155
epoch 8, train RMSE: 0.7102
epoch 9, train RMSE: 0.7063
epoch 10, train RMSE: 0.7026
epoch 11, train RMSE: 0.6993
epoch 12, train RMSE: 0.6963
epoch 13, train RMSE: 0.6941
epoch 14, train RMSE: 0.6922
epoch 15, train RMSE: 0.6904
epoch 16, train RMSE: 0.6887
epoch 17, train RMSE: 0.6875
epoch 18, train RMSE: 0.6866
epoch 19, train RMSE: 0.6855
epoch 20, train RMSE: 0.6847


In [80]:
from mxnet import nd

def evaluate_rmse(net, data_iter, ctx):
    # L2 loss fonksiyonu RMSE hesaplaması için uygundur
    loss = gluon.loss.L2Loss()

    # Toplam kayıp ve örnek sayısını biriktiren yardımcı bir araç
    metric = d2l.Accumulator(2)

    for i, (customers, movies, rates) in enumerate(data_iter):  # Veri kümesindeki her batch için döngü
        customers, movies, rates = customers.as_in_ctx(ctx), movies.as_in_ctx(ctx), rates.as_in_ctx(ctx)

        # Verileri belirtilen cihazda (CPU/GPU) kullanmak için uygun forma dönüştür
        preds = net(customers, movies)  # Modeli kullanarak kullanıcı ve öğe çiftleri için tahminler al
        l = loss(preds, rates)  # Tahminler ile gerçek puanlar arasındaki kaybı (L2Loss) hesapla
        metric.add(l.sum().item(), rates.size)  # Toplam kaybı ve işlenen örnek sayısını biriktir

    # Ortalama kare hatayı hesapla (karekök al)
    rmse = nd.sqrt(nd.array(metric[0]) / nd.array(metric[1]))
    return rmse.asscalar()

test_rmse = evaluate_rmse(net, test_iter, ctx)  # Test veri kümesi üzerinde RMSE hesapla
print(f'Test RMSE: {test_rmse:.4f}')  # Test RMSE değerini yazdır

Test RMSE: 0.7612
